# JN12 — DTM Format Conversion for Tsunami-HySEA
## Converting GeoTIFF, Surfer Binary v6 and v7 to HySEA `.grd`

**HySEALab · Preprocessing Notebooks · EDANYA Research Group, Universidad de Málaga**
*Edited by José Manuel González Vida*

---

Tsunami-HySEA requires bathymetric/topographic grids in a specific **NetCDF4 format**
with extension `.grd` and three variables named exactly `x`, `y`, and `z`.

In practice, elevation data arrives in many different formats depending on the source:

| Format | Extension | Magic bytes | Typical source |
|--------|-----------|-------------|----------------|
| **GeoTIFF** | `.tif` / `.tiff` | `II*\x00` or `MM\x00*` | GEBCO, Copernicus DEM, IGN LiDAR, local surveys |
| **Surfer Binary v6** | `.grd` | `DSBB` | Golden Software Surfer exports, IHO data |
| **Surfer Binary v7** | `.grd` | `DSRB` | Golden Software Surfer 7+ exports |
| **HySEA / GMT NetCDF4** | `.grd` | (NetCDF magic) | Already compatible — no conversion needed |

---

### Requirements

**Python packages** (any recent version):

```bash
conda install -c conda-forge numpy matplotlib pillow netcdf4
# or: pip install numpy matplotlib pillow netCDF4
```

**Optional command-line tools** (only for Sections F and G):

- **GMT 6** — [generic-mapping-tools.org/download](https://www.generic-mapping-tools.org/download/)
- **GDAL** (`gdal_translate`) — `conda install -c conda-forge gdal`

**Input data:** your own DTM files (GeoTIFF or Surfer grids) placed in the
`data/` folder next to this notebook. **No input data is strictly required** —
Sections B and C generate synthetic Surfer test files, so the notebook can be
executed end-to-end out of the box.

---

### What this notebook does

```
your_dem.tif          ─┐
your_grid_v6.grd      ─┼──[convert_to_hysea_grd()]──► hysea_ready.grd
your_grid_v7.grd      ─┘
```

| Section | Content |
|---------|---------|
| A | GeoTIFF → HySEA `.grd` |
| B | Surfer Binary v6 → HySEA `.grd` |
| C | Surfer Binary v7 → HySEA `.grd` |
| D | Auto-detection + unified `convert_to_hysea_grd()` function |
| E | Batch conversion (loop over a folder) |
| F | GMT command-line alternative |
| G | **Foolproof route**: any format → TIF → HySEA `.grd` |

> **See also:** [JN04](JN04_Grid_from_GEBCO.ipynb) covers building a HySEA grid
> directly from GEBCO (GeoTIFF), including clipping to a bounding box and
> resampling to a target resolution.
> This notebook focuses on **format conversion** without changing resolution.

---
## The HySEA `.grd` Format

HySEA requires **NetCDF4** files with exactly these variables:

| Variable | Dimension | Type | Description |
|----------|-----------|------|-------------|
| `x` | `(nx,)` | float64 | Longitude or Easting vector (**ascending**) |
| `y` | `(ny,)` | float64 | Latitude or Northing vector (**ascending**, S→N) |
| `z` | `(ny, nx)` | float32 | Elevation / depth in metres (ocean = **negative**) |

### Sign convention

HySEA uses the **GMT convention**: ocean depths are negative, land elevations are positive.
Most bathymetric datasets (GEBCO, EMODnet) already follow this convention.
Some local surveys deliver bathymetry as positive depth — use `z = -z` to flip the sign.

### Coordinate system

HySEA works in **geographic coordinates** (decimal degrees, WGS84).
If your DTM is in a projected system (UTM, Lambert, …), you must reproject it to
lat/lon before conversion.  This notebook assumes geographic coordinates.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 1 — Libraries
# ─────────────────────────────────────────────────────────────────────────────

import copy
import os
import struct
import subprocess

import numpy as np
import matplotlib.pyplot as plt

from netCDF4 import Dataset
from datetime import datetime
from PIL import Image

Image.MAX_IMAGE_PIXELS = None   # disable PIL size limit for large GeoTIFFs

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size']  = 11

# ── Folder for input and output files ────────────────────────────────────────
DATA_DIR = 'data'               # relative to this notebook
os.makedirs(DATA_DIR, exist_ok=True)

print('Libraries loaded OK')
print(f'Working directory : {os.getcwd()}')
print(f'Data directory    : {os.path.abspath(DATA_DIR)}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 2 — Core utilities: grdwrite, grdread, plot_grd, verify_grd
# ─────────────────────────────────────────────────────────────────────────────

def grdwrite(x, y, z, foutput):
    """
    Write a HySEA-compatible NetCDF4 .grd file.
    Variables: x (lon, f64), y (lat, f64), z (depth/elev, f32).
    Warns if z contains NaN — HySEA cannot run with gaps in the bathymetry.
    """
    n_nan = int(np.isnan(z).sum())
    if n_nan:
        print(f'⚠️  WARNING: {os.path.basename(foutput)} contains {n_nan} NaN cell(s).')
        print('   HySEA cannot run with NaN in the bathymetry — fill the gaps')
        print('   (e.g. nearest-neighbour interpolation) before using this grid.')
    ds = Dataset(foutput, 'w', format='NETCDF4')
    ds.createDimension('x', len(x))
    ds.createDimension('y', len(y))
    vx = ds.createVariable('x', 'f8', 'x')
    vy = ds.createVariable('y', 'f8', 'y')
    vz = ds.createVariable('z', 'f4', ('y', 'x'))
    vx[:] = x;  vy[:] = y;  vz[:, :] = z
    vx.units = 'degrees_east'
    vy.units = 'degrees_north'
    vz.units = 'meters'
    ds.title       = os.path.basename(foutput)
    ds.history     = 'Created by JN12_Format_Conversion.ipynb'
    ds.description = 'Converted ' + datetime.today().strftime('%d/%m/%Y')
    ds.close()


def grdread(fpath):
    """
    Read a HySEA .grd (NetCDF4) file.
    Returns x (1-D), y (1-D), z (2-D), dx, dy.
    """
    ds = Dataset(fpath)
    x = y = z = None
    for vn in ds.variables:
        if vn in ('x', 'lon', 'longitude'): x = np.array(ds[vn][:])
        if vn in ('y', 'lat', 'latitude'):  y = np.array(ds[vn][:])
        if vn in ('z', 'topo', 'Band1'):    z = np.array(ds[vn][:])
    ds.close()
    return x, y, z, float(x[1]-x[0]), float(y[1]-y[0])


def plot_grd(fpath, title=None, vmin=-3000, vmax=1000, figsize=(9, 6)):
    """Quick plot of a HySEA .grd file."""
    x, y, z, dx, dy = grdread(fpath)
    fig, ax = plt.subplots(figsize=figsize, constrained_layout=True)
    cm = copy.copy(plt.get_cmap('terrain'))
    cm.set_bad('white', 0.0)
    im = ax.pcolormesh(x, y, z, cmap=cm, shading='auto', vmin=vmin, vmax=vmax)
    ax.contour(x, y, z, levels=[0], colors='black', linewidths=0.8, alpha=0.8)
    fig.colorbar(im, ax=ax, pad=0.02, label='Depth / Elevation (m)')
    if title is None:
        title = (f'{os.path.basename(fpath)}\n'
                 f'{len(x)}×{len(y)} cells  ·  Δx≈{dx*111320:.0f} m  ·  '
                 f'z=[{float(np.nanmin(z)):.0f}, {float(np.nanmax(z)):.0f}] m')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Longitude (°E)', fontsize=9)
    ax.set_ylabel('Latitude (°N)', fontsize=9)
    ax.grid(True, linestyle='--', alpha=0.3)
    plt.show()


def verify_grd(fpath):
    """Print a verification summary for a HySEA .grd file."""
    x, y, z, dx, dy = grdread(fpath)
    nan_n = int(np.isnan(z).sum())
    print(f'  File       : {os.path.basename(fpath)}')
    print(f'  nx × ny    : {len(x)} × {len(y)} = {len(x)*len(y):,} cells')
    print(f'  Lon        : [{float(x[0]):.4f}, {float(x[-1]):.4f}] °E')
    print(f'  Lat        : [{float(y[0]):.4f}, {float(y[-1]):.4f}] °N')
    print(f'  dx         : {dx:.6f}°  ≈  {dx*111320:.0f} m')
    print(f'  z range    : [{float(np.nanmin(z)):.1f}, {float(np.nanmax(z)):.1f}] m')
    print(f'  NaN cells  : {nan_n}  {"✅" if nan_n == 0 else "⚠️  check coverage"}')
    print()


print('Utilities defined OK')

---
## Section A — GeoTIFF → HySEA `.grd`

A **GeoTIFF** embeds georeferencing information in TIFF tags:

| TIFF tag | Hex | Content |
|----------|-----|---------|
| `ModelPixelScaleTag` | `0x830E` (33550) | `(dx, dy, 0)` — pixel size in degrees |
| `ModelTiepointTag` | `0x8482` (33922) | `(i, j, k, lon_UL, lat_UL, 0)` — UL corner |

The pixel at row 0, col 0 is the **upper-left** (north-west) corner.  Rows increase
southward, so latitudes are **descending** — we flip them to ascending before writing.

> **Full workflow** (clip + resample + write): see **JN04**.  
> This section only covers the simplest case: read the TIF **as-is** and write it to `.grd`
> without changing resolution or extent.

### NoData values in GeoTIFF

Some GeoTIFFs encode missing/land cells with a sentinel value (−9999, −32768, 3.4×10³⁸, …).
Specify the sentinel in `nodata=` so it is replaced by `NaN` before writing.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 3 — read_geotiff()
# ─────────────────────────────────────────────────────────────────────────────

def read_geotiff(filepath, nodata=None):
    """
    Read a geographic GeoTIFF (WGS84 lat/lon) and return HySEA-ready arrays.

    Parameters
    ----------
    filepath : str   — path to the .tif file
    nodata   : float — NoData sentinel to replace with NaN (None = skip)

    Returns
    -------
    x  : 1-D float64 array — longitude vector (ascending, west→east)
    y  : 1-D float64 array — latitude vector  (ascending, south→north)
    z  : 2-D float32 array — depth/elevation, shape (ny, nx)
    dx : float — longitude cell spacing (degrees)
    dy : float — latitude  cell spacing (degrees)
    """
    img = Image.open(filepath)
    tv2 = img.tag_v2
    w, h = img.size

    # Extract georeferencing from TIFF tags
    if 33550 not in tv2 or 33922 not in tv2:
        raise ValueError(
            f'{filepath}: GeoTIFF tags 33550/33922 not found.\n'
            'The file may not be georeferenced, or may use a different tag layout.')

    dx   = float(tv2[33550][0])     # longitude pixel size
    dy   = float(tv2[33550][1])     # latitude  pixel size (positive)
    lon0 = float(tv2[33922][3])     # longitude of upper-left pixel centre
    lat0 = float(tv2[33922][4])     # latitude  of upper-left pixel centre

    # Warn if coordinates look like a projected system (metres, not degrees)
    if abs(lon0) > 180 or abs(lat0) > 90:
        print('⚠️  WARNING: coordinates look like a projected system '
              f'(lon0={lon0:.1f}, lat0={lat0:.1f}).\n'
              '   HySEA requires geographic coordinates (WGS84 degrees).\n'
              '   Reproject to lat/lon before using this file in HySEA.')

    # Read pixel data
    z = np.array(img, dtype=np.float32)
    img.close()

    # Build coordinate axes
    x = lon0 + np.arange(w) * dx                  # ascending west→east
    y = lat0 - np.arange(h) * dy                  # descending north→south

    # Flip to ascending latitude (required by HySEA / RegularGridInterpolator)
    y = y[::-1].copy()
    z = z[::-1, :].copy()

    # Replace NoData sentinel
    if nodata is not None:
        z[z == nodata] = np.nan

    return x.astype(np.float64), y.astype(np.float64), z, dx, dy


print('read_geotiff() defined OK')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 4 — Example: convert a GeoTIFF to HySEA .grd
# Replace TIF_INPUT with the path to your .tif file
# ─────────────────────────────────────────────────────────────────────────────

# ── User settings ─────────────────────────────────────────────────────────────
TIF_INPUT  = os.path.join(DATA_DIR, 'your_dem.tif')      # ← replace with your file
TIF_OUTPUT = os.path.join(DATA_DIR, 'your_dem_hysea.grd')
TIF_NODATA = None   # e.g. -9999 or -32768 — set to None if not applicable

# ── Conversion ────────────────────────────────────────────────────────────────
if os.path.exists(TIF_INPUT):
    print(f'Reading: {TIF_INPUT}')
    x, y, z, dx, dy = read_geotiff(TIF_INPUT, nodata=TIF_NODATA)

    print(f'  Grid : {len(x)} × {len(y)} cells')
    print(f'  Lon  : [{float(x[0]):.4f}, {float(x[-1]):.4f}] °E')
    print(f'  Lat  : [{float(y[0]):.4f}, {float(y[-1]):.4f}] °N')
    print(f'  dx   : {dx:.6f}°  ≈  {dx*111320:.0f} m')
    print(f'  z    : [{float(np.nanmin(z)):.1f}, {float(np.nanmax(z)):.1f}] m')
    print()

    grdwrite(x, y, z, TIF_OUTPUT)
    size_mb = os.path.getsize(TIF_OUTPUT) / 1024**2
    print(f'Written: {TIF_OUTPUT}  ({size_mb:.1f} MB)')
    print()
    verify_grd(TIF_OUTPUT)
    plot_grd(TIF_OUTPUT)
else:
    print(f'File not found: {TIF_INPUT}')
    print('Set TIF_INPUT to your GeoTIFF path and re-run this cell.')
    print()
    print('Example using the GEBCO file from JN04:')
    print("  TIF_INPUT  = os.path.join(DATA_DIR, 'gebco_2025_sub_ice_n90.0_s0.0_w0.0_e90.0.tif')")
    print('  TIF_NODATA = None')

---
## Section B — Surfer Binary v6 → HySEA `.grd`

Golden Software **Surfer 6 Binary** grid files start with the 4-byte magic `DSBB`.

### File structure

```
Offset  Size  Type       Field
──────  ────  ─────────  ──────────────────────────────────────────
     0     4  char[4]    Magic = "DSBB"
     4     2  uint16     ncols — number of columns (x direction)
     6     2  uint16     nrows — number of rows    (y direction)
     8     8  float64    xmin  — western  edge (centre of first column)
    16     8  float64    xmax  — eastern  edge
    24     8  float64    ymin  — southern edge (centre of first row stored)
    32     8  float64    ymax  — northern edge
    40     8  float64    zmin  — minimum z value (for verification)
    48     8  float64    zmax  — maximum z value
    56   4×nrows×ncols  float32  z data  (row 0 = south, row nrows-1 = north)
```

All multi-byte values are **little-endian**.

### Blank (NoData) value

Surfer uses `1.701410009187828 × 10³⁸` (≈ `float32` maximum) as the blank sentinel.
The reader replaces any value ≥ 99% of this constant with `NaN`.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 5 — read_surfer6_binary()
# ─────────────────────────────────────────────────────────────────────────────

SURFER_BLANK = 1.701410009187828e+38   # Surfer blank / NoData sentinel


def read_surfer6_binary(filepath, nodata=None):
    """
    Read a Surfer 6 binary grid (.grd, magic 'DSBB').

    Parameters
    ----------
    filepath : str   — path to the Surfer 6 .grd file
    nodata   : float — additional NoData sentinel (None = use only Surfer blank)

    Returns
    -------
    x, y : 1-D float64 arrays — coordinate vectors (ascending)
    z    : 2-D float64 array  — depth/elevation, shape (nrows, ncols)
    dx, dy : float — cell spacings
    """
    with open(filepath, 'rb') as f:
        # ── Header ──────────────────────────────────────────────────────────
        magic = f.read(4)
        if magic != b'DSBB':
            raise ValueError(
                f'{filepath}: expected magic b"DSBB", got {magic!r}.\n'
                'Check the file format (Surfer 7 uses "DSRB").')

        ncols, nrows = struct.unpack('<HH', f.read(4))
        xmin, xmax, ymin, ymax, zmin, zmax = struct.unpack('<6d', f.read(48))

        # ── Data ────────────────────────────────────────────────────────────
        raw = f.read(nrows * ncols * 4)   # float32, 4 bytes per value
        z = np.frombuffer(raw, dtype='<f4').reshape(nrows, ncols).astype(np.float64)

    # Replace Surfer blank + custom nodata with NaN
    z[z >= SURFER_BLANK * 0.99] = np.nan
    if nodata is not None:
        z[z == float(nodata)] = np.nan

    # Build coordinate axes
    if ncols > 1:
        dx = (xmax - xmin) / (ncols - 1)
        x  = np.linspace(xmin, xmax, ncols)
    else:
        dx = 0.0;  x = np.array([xmin])

    if nrows > 1:
        dy = (ymax - ymin) / (nrows - 1)
        y  = np.linspace(ymin, ymax, nrows)   # ascending south → north
    else:
        dy = 0.0;  y = np.array([ymin])

    return x, y, z, dx, dy


print('read_surfer6_binary() defined OK')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 6a — Create a synthetic Surfer 6 file for testing
#           (skip this cell if you have a real file)
# ─────────────────────────────────────────────────────────────────────────────

def write_surfer6_binary(x, y, z, filepath):
    """
    Write arrays as a Surfer 6 binary .grd file (useful for testing / export).
    NaN values are encoded as the Surfer blank sentinel.
    """
    nrows, ncols = z.shape
    assert len(x) == ncols and len(y) == nrows, 'Shape mismatch'

    z_out = z.astype(np.float32).copy()
    z_out[np.isnan(z_out)] = np.float32(SURFER_BLANK)
    valid = z[~np.isnan(z)]
    zmin = float(valid.min()) if len(valid) else 0.0
    zmax = float(valid.max()) if len(valid) else 0.0

    with open(filepath, 'wb') as f:
        f.write(b'DSBB')
        f.write(struct.pack('<HH', ncols, nrows))
        f.write(struct.pack('<6d', x[0], x[-1], y[0], y[-1], zmin, zmax))
        f.write(z_out.tobytes())


# Generate a synthetic 50×60 grid of Catania area bathymetry
SYNTH_S6 = os.path.join(DATA_DIR, 'synth_surfer6.grd')

xs = np.linspace(15.00, 15.20, 60)
ys = np.linspace(37.30, 37.52, 50)
XS, YS = np.meshgrid(xs, ys)
# Synthetic bathymetry: deeper offshore, shallow coast, some land
zs = -2000 + 2500 * np.sqrt((XS - 15.00)**2 / 0.04 + (YS - 37.30)**2 / 0.04)
zs = np.clip(zs, -2000, 500)

write_surfer6_binary(xs, ys, zs.astype(np.float32), SYNTH_S6)
print(f'Synthetic Surfer 6 file written: {SYNTH_S6}')
print(f'  ncols={len(xs)}, nrows={len(ys)},  z=[{zs.min():.0f}, {zs.max():.0f}] m')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 6b — Example: convert a Surfer 6 file to HySEA .grd
# Replace S6_INPUT with your actual file path
# ─────────────────────────────────────────────────────────────────────────────

# ── User settings ─────────────────────────────────────────────────────────────
S6_INPUT  = SYNTH_S6                                          # ← replace with your file
# S6_INPUT  = os.path.join(DATA_DIR, 'your_surfer6.grd')     # ← or point to your file
S6_OUTPUT = os.path.join(DATA_DIR, 'your_surfer6_hysea.grd')
S6_NODATA = None   # extra NoData value (e.g. -9999) — None to skip

# ── Conversion ────────────────────────────────────────────────────────────────
print(f'Reading Surfer 6 file: {S6_INPUT}')
x, y, z, dx, dy = read_surfer6_binary(S6_INPUT, nodata=S6_NODATA)

print(f'  ncols × nrows : {len(x)} × {len(y)}')
print(f'  Lon           : [{float(x[0]):.4f}, {float(x[-1]):.4f}]')
print(f'  Lat           : [{float(y[0]):.4f}, {float(y[-1]):.4f}]')
print(f'  dx            : {dx:.6f}°  ≈  {dx*111320:.0f} m')
print(f'  z range       : [{float(np.nanmin(z)):.1f}, {float(np.nanmax(z)):.1f}] m')
print(f'  NaN cells     : {int(np.isnan(z).sum())}')
print()

grdwrite(x, y, z, S6_OUTPUT)
size_mb = os.path.getsize(S6_OUTPUT) / 1024**2
print(f'Written: {S6_OUTPUT}  ({size_mb:.2f} MB)')
print()
verify_grd(S6_OUTPUT)
plot_grd(S6_OUTPUT, title=f'Surfer 6 → HySEA .grd  ({os.path.basename(S6_INPUT)})')

---
## Section C — Surfer Binary v7 → HySEA `.grd`

**Surfer 7 Binary** uses a **section-based** format (magic `DSRB`).
Each section starts with a 4-byte tag + 4-byte size, followed by the section data.

### Section layout

```
Tag     Size  Content
──────  ────  ─────────────────────────────────────────────────────────
"DSRB"     8  int32 version (=1)  +  int32 reserved (=1)
"GRID"    72  2×int32 + 8×float64  (grid parameters, see below)
"DATA"    8×nrow×ncol  float64 z values  (row 0 = ymin = south)
"FITA"   (optional fault/break-line data — ignored here)
```

### GRID section parameters (72 bytes)

```
nrow    int32   — number of rows (y direction)
ncol    int32   — number of columns (x direction)
xll     f64     — x coordinate of lower-left corner
yll     f64     — y coordinate of lower-left corner
xsize   f64     — x cell spacing
ysize   f64     — y cell spacing
zmin    f64     — minimum z
zmax    f64     — maximum z
rot     f64     — rotation angle (usually 0)
blank   f64     — NoData sentinel value (≈ 1.701×10³⁸)
```

### Differences from v6

| Property | Surfer v6 | Surfer v7 |
|----------|-----------|-----------|
| Magic bytes | `DSBB` | `DSRB` |
| z data type | float32 | **float64** |
| Max grid size | 65535 × 65535 | unlimited |
| Coordinate precision | double | double |
| Section structure | linear | **tagged sections** |

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 7 — read_surfer7_binary()
# ─────────────────────────────────────────────────────────────────────────────

def read_surfer7_binary(filepath, nodata=None):
    """
    Read a Surfer 7 binary grid (.grd, magic 'DSRB').

    Parameters
    ----------
    filepath : str   — path to the Surfer 7 .grd file
    nodata   : float — additional NoData sentinel (None = use only Surfer blank)

    Returns
    -------
    x, y : 1-D float64 arrays — coordinate vectors (ascending)
    z    : 2-D float64 array  — depth/elevation, shape (nrows, ncols)
    dx, dy : float — cell spacings
    """
    nrow = ncol = None
    x0 = y0 = dx = dy = None
    blank = SURFER_BLANK
    z = None

    with open(filepath, 'rb') as f:
        # First check magic
        magic = f.read(4)
        if magic != b'DSRB':
            raise ValueError(
                f'{filepath}: expected magic b"DSRB", got {magic!r}.\n'
                'Check the file format (Surfer 6 uses "DSBB").')
        f.seek(0)   # rewind; the magic is the first section tag

        # ── Parse sections ────────────────────────────────────────────────────
        while True:
            buf = f.read(8)
            if len(buf) < 8:
                break
            tag  = buf[:4]                               # 4-byte ASCII tag
            size = struct.unpack('<i', buf[4:8])[0]      # section data size (bytes)
            content = f.read(size)

            if tag == b'GRID':
                # 2×int32 + 8×float64 = 8 + 64 = 72 bytes
                nrow, ncol = struct.unpack('<2i', content[:8])
                x0, y0, dx, dy, zmin_h, zmax_h, rot, blank = \
                    struct.unpack('<8d', content[8:72])

            elif tag == b'DATA':
                # float64 values; nrow × ncol
                expected = nrow * ncol * 8
                if len(content) < expected:
                    raise ValueError(
                        f'DATA section is too short: '
                        f'got {len(content)} bytes, expected {expected}')
                z = np.frombuffer(content[:expected],
                                   dtype='<f8').reshape(nrow, ncol).copy()
            # Other sections (DSRB, FITA, …) are silently skipped

    if z is None:
        raise ValueError('No DATA section found in the Surfer 7 file.')
    if nrow is None:
        raise ValueError('No GRID section found in the Surfer 7 file.')

    # Replace Surfer blank + custom nodata with NaN
    z[z >= blank * 0.99] = np.nan
    if nodata is not None:
        z[z == float(nodata)] = np.nan

    # Build coordinate axes from lower-left corner + spacing
    x = x0 + np.arange(ncol) * dx   # ascending west → east
    y = y0 + np.arange(nrow) * dy   # ascending south → north

    return x, y, z, dx, dy


print('read_surfer7_binary() defined OK')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 8a — Create a synthetic Surfer 7 file for testing
#           (skip this cell if you have a real file)
# ─────────────────────────────────────────────────────────────────────────────

def write_surfer7_binary(x, y, z, filepath):
    """
    Write arrays as a Surfer 7 binary .grd file.
    NaN values are encoded as the Surfer blank sentinel.
    """
    nrow, ncol = len(y), len(x)
    z_out = z.astype(np.float64).copy()
    z_out[np.isnan(z_out)] = SURFER_BLANK
    valid = z[~np.isnan(z)]
    zmin = float(valid.min()) if len(valid) else 0.0
    zmax = float(valid.max()) if len(valid) else 0.0
    x0, y0 = float(x[0]), float(y[0])
    dx = float(x[1] - x[0]) if len(x) > 1 else 1.0
    dy = float(y[1] - y[0]) if len(y) > 1 else 1.0

    grid_content = struct.pack('<2i', nrow, ncol) + \
                   struct.pack('<8d', x0, y0, dx, dy, zmin, zmax, 0.0, SURFER_BLANK)
    data_content = z_out.tobytes()

    with open(filepath, 'wb') as f:
        # DSRB section
        f.write(b'DSRB')
        f.write(struct.pack('<i', 8))
        f.write(struct.pack('<2i', 1, 1))   # version=1, reserved=1
        # GRID section
        f.write(b'GRID')
        f.write(struct.pack('<i', len(grid_content)))
        f.write(grid_content)
        # DATA section
        f.write(b'DATA')
        f.write(struct.pack('<i', len(data_content)))
        f.write(data_content)


# Generate a synthetic Surfer 7 file (same domain, float64 precision)
SYNTH_S7 = os.path.join(DATA_DIR, 'synth_surfer7.grd')
write_surfer7_binary(xs, ys, zs, SYNTH_S7)
print(f'Synthetic Surfer 7 file written: {SYNTH_S7}')
print(f'  ncols={len(xs)}, nrows={len(ys)},  z=[{zs.min():.0f}, {zs.max():.0f}] m')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 8b — Example: convert a Surfer 7 file to HySEA .grd
# Replace S7_INPUT with your actual file path
# ─────────────────────────────────────────────────────────────────────────────

# ── User settings ─────────────────────────────────────────────────────────────
S7_INPUT  = SYNTH_S7                                          # ← replace with your file
# S7_INPUT  = os.path.join(DATA_DIR, 'your_surfer7.grd')     # ← or point to your file
S7_OUTPUT = os.path.join(DATA_DIR, 'your_surfer7_hysea.grd')
S7_NODATA = None

# ── Conversion ────────────────────────────────────────────────────────────────
print(f'Reading Surfer 7 file: {S7_INPUT}')
x, y, z, dx, dy = read_surfer7_binary(S7_INPUT, nodata=S7_NODATA)

print(f'  ncols × nrows : {len(x)} × {len(y)}')
print(f'  Lon           : [{float(x[0]):.4f}, {float(x[-1]):.4f}]')
print(f'  Lat           : [{float(y[0]):.4f}, {float(y[-1]):.4f}]')
print(f'  dx            : {dx:.6f}°  ≈  {dx*111320:.0f} m')
print(f'  z range       : [{float(np.nanmin(z)):.1f}, {float(np.nanmax(z)):.1f}] m')
print(f'  NaN cells     : {int(np.isnan(z).sum())}')
print()

grdwrite(x, y, z, S7_OUTPUT)
size_mb = os.path.getsize(S7_OUTPUT) / 1024**2
print(f'Written: {S7_OUTPUT}  ({size_mb:.2f} MB)')
print()
verify_grd(S7_OUTPUT)
plot_grd(S7_OUTPUT, title=f'Surfer 7 → HySEA .grd  ({os.path.basename(S7_INPUT)})')

---
## Section D — Auto-detection and Unified Converter

The `detect_format()` function inspects the first 4 bytes of the file to identify
the format.  The `convert_to_hysea_grd()` function wraps all three readers
into a single call.

| First 4 bytes | Format |
|---------------|--------|
| `DSBB` | Surfer Binary v6 |
| `DSRB` | Surfer Binary v7 |
| `II*\x00` or `MM\x00*` | GeoTIFF (little- or big-endian) |
| `\x89HDF` | HDF5 / NetCDF4 (already HySEA-compatible) |

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 9 — detect_format() + convert_to_hysea_grd()
# ─────────────────────────────────────────────────────────────────────────────

TIFF_LE_MAGIC = b'II*\x00'    # little-endian TIFF
TIFF_BE_MAGIC = b'MM\x00*'    # big-endian TIFF
HDF5_MAGIC    = b'\x89HDF'    # HDF5 / NetCDF4


def detect_format(filepath):
    """
    Detect the format of a DTM file by inspecting the first 4 magic bytes.

    Returns
    -------
    str : 'geotiff' | 'surfer6' | 'surfer7' | 'netcdf' | 'unknown'
    """
    with open(filepath, 'rb') as f:
        magic = f.read(4)

    if magic == b'DSBB':
        return 'surfer6'
    if magic == b'DSRB':
        return 'surfer7'
    if magic in (TIFF_LE_MAGIC, TIFF_BE_MAGIC):
        return 'geotiff'
    if magic == HDF5_MAGIC:
        return 'netcdf'
    return 'unknown'


def convert_to_hysea_grd(input_file, output_file,
                          nodata=None, flip_sign=False, verbose=True):
    """
    Convert any supported DTM format to a HySEA-compatible NetCDF4 .grd file.

    Parameters
    ----------
    input_file  : str   — path to the source DTM (.tif, Surfer .grd, …)
    output_file : str   — path for the HySEA .grd output
    nodata      : float — extra NoData sentinel to replace with NaN
    flip_sign   : bool  — if True, z = -z (for datasets where depth is positive)
    verbose     : bool  — print a summary after conversion

    Returns
    -------
    fmt : str — detected format string
    """
    fmt = detect_format(input_file)

    if fmt == 'geotiff':
        x, y, z, dx, dy = read_geotiff(input_file, nodata=nodata)
    elif fmt == 'surfer6':
        x, y, z, dx, dy = read_surfer6_binary(input_file, nodata=nodata)
    elif fmt == 'surfer7':
        x, y, z, dx, dy = read_surfer7_binary(input_file, nodata=nodata)
    elif fmt == 'netcdf':
        print(f'  {os.path.basename(input_file)}: already NetCDF4 — '
              'copying directly.')
        import shutil
        shutil.copy2(input_file, output_file)
        return fmt
    else:
        raise ValueError(
            f'Unsupported format for {input_file}\n'
            'Supported: GeoTIFF (.tif), Surfer Binary v6/v7 (.grd), NetCDF4 (.grd)')

    if flip_sign:
        z = -z

    grdwrite(x, y, z, output_file)

    if verbose:
        size_mb = os.path.getsize(output_file) / 1024**2
        nan_n   = int(np.isnan(z).sum())
        print(f'  [{fmt:8s}]  {os.path.basename(input_file)}')
        print(f'             → {os.path.basename(output_file)}')
        print(f'             {len(x)}×{len(y)} cells  ·  '
              f'Δx≈{dx*111320:.0f} m  ·  '
              f'z=[{float(np.nanmin(z)):.0f},{float(np.nanmax(z)):.0f}] m  ·  '
              f'{size_mb:.2f} MB  ·  NaN={nan_n}')

    return fmt


print('detect_format() and convert_to_hysea_grd() defined OK')

# ── Quick test with the synthetic files ───────────────────────────────────────
print()
print('FORMAT DETECTION TEST')
for f in [SYNTH_S6, SYNTH_S7]:
    print(f'  {os.path.basename(f):30s} → {detect_format(f)}')

---
## Section E — Batch Conversion

When you have many files in a folder (e.g. DEM tiles `dom1.grd`, `dom2.grd`, …),
convert all of them in a single loop.

The output files are placed in the same folder with `_hysea` appended to the stem.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 10 — Batch conversion
# ─────────────────────────────────────────────────────────────────────────────

# ── User settings ─────────────────────────────────────────────────────────────
INPUT_FOLDER  = DATA_DIR          # ← folder containing your source files
OUTPUT_FOLDER = DATA_DIR          # ← folder for HySEA .grd outputs
EXTENSIONS    = ('.tif', '.tiff', '.grd')   # file extensions to process
NODATA_VALUE  = None              # shared NoData sentinel (or None)
FLIP_SIGN     = False             # True if your bathymetry is stored as positive depth

# Exclude files we already know are HySEA outputs
SKIP_SUFFIX   = '_hysea.grd'

# ── Scan and convert ──────────────────────────────────────────────────────────
candidates = sorted([
    f for f in os.listdir(INPUT_FOLDER)
    if any(f.lower().endswith(ext) for ext in EXTENSIONS)
    and not f.endswith(SKIP_SUFFIX)
])

print(f'Found {len(candidates)} candidate file(s) in {INPUT_FOLDER}')
print()

converted = []
skipped   = []
errors    = []

for fname in candidates:
    in_path  = os.path.join(INPUT_FOLDER, fname)
    stem     = os.path.splitext(fname)[0]
    out_path = os.path.join(OUTPUT_FOLDER, stem + '_hysea.grd')

    fmt = detect_format(in_path)
    if fmt == 'unknown':
        skipped.append(fname)
        print(f'  SKIP  {fname}  (unknown format)')
        continue
    if fmt == 'netcdf':
        skipped.append(fname)
        print(f'  SKIP  {fname}  (already NetCDF4 — no conversion needed)')
        continue

    try:
        convert_to_hysea_grd(in_path, out_path,
                              nodata=NODATA_VALUE,
                              flip_sign=FLIP_SIGN,
                              verbose=True)
        converted.append(out_path)
    except Exception as exc:
        errors.append((fname, str(exc)))
        print(f'  ERROR {fname}: {exc}')

print()
print(f'Converted : {len(converted)}')
print(f'Skipped   : {len(skipped)}')
print(f'Errors    : {len(errors)}')

---
## Section F — GMT Command-Line Alternative

If **GMT 6** is available, `gmt grdconvert` can convert many formats to NetCDF4
in a single shell command:

```bash
# GeoTIFF → HySEA-compatible NetCDF4
gmt grdconvert dom$i.tif -Gdom$i.grd=nf

# Surfer Binary (.grd) → NetCDF4
gmt grdconvert surfer_input.grd -Goutput.grd=nf

# Batch: convert all .tif files in current folder
for f in *.tif; do
    base=${f%.tif}
    gmt grdconvert "$f" -G"${base}.grd=nf"
done
```

The `=nf` suffix is the GMT format code for **NetCDF4 float32**.
GMT writes variables named `x`, `y`, `z` — exactly what HySEA expects.

> **Note:** GMT's `grdconvert` handles dozens of formats automatically
> (NetCDF3/4, GeoTIFF, Surfer, ESRI, ERDAS, …).  For simple single-file
> conversions on a system where GMT is installed, it is the fastest option.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 11 — GMT grdconvert via Python subprocess
# ─────────────────────────────────────────────────────────────────────────────

def gmt_convert(input_file, output_file, fmt_code='nf'):
    """
    Convert a DTM file to HySEA .grd using GMT grdconvert.

    Parameters
    ----------
    input_file  : str — source file (.tif, Surfer .grd, …)
    output_file : str — output file path (will be written as NetCDF4)
    fmt_code    : str — GMT format suffix (default 'nf' = NetCDF4 float32)
    """
    target = f'{output_file}={fmt_code}'
    cmd = ['gmt', 'grdconvert', input_file, f'-G{target}']

    print(f'Running: {" ".join(cmd)}')
    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        print(f'  GMT error:\n{result.stderr}')
        return False

    size_mb = os.path.getsize(output_file) / 1024**2
    print(f'  ✅  Written: {output_file}  ({size_mb:.2f} MB)')
    return True


# ── Check if GMT is available ─────────────────────────────────────────────────
gmt_check = subprocess.run(['gmt', '--version'],
                           capture_output=True, text=True)
if gmt_check.returncode == 0:
    print(f'GMT version: {gmt_check.stdout.strip()}')
    print('GMT is available — you can use gmt_convert() or run shell commands.')
    print()

    # Example — uncomment and set your file paths:
    # gmt_convert(
    #     input_file  = os.path.join(DATA_DIR, 'dom1.tif'),
    #     output_file = os.path.join(DATA_DIR, 'dom1.grd'),
    #     fmt_code    = 'nf'
    # )

    # Batch example (same as the shell loop above):
    # for i in range(1, 6):
    #     gmt_convert(
    #         input_file  = os.path.join(DATA_DIR, f'dom{i}.tif'),
    #         output_file = os.path.join(DATA_DIR, f'dom{i}.grd'),
    #     )
else:
    print('GMT is not installed or not in PATH.')
    print('Install GMT 6: https://www.generic-mapping-tools.org/download/')
    print()
    print('Alternative shell command (no Python):')
    print("  for f in *.tif; do")
    print("      gmt grdconvert \"$f\" -G\"${f%.tif}.grd=nf\"")
    print("  done")

---
## Section G — The Foolproof Route: Any Format → TIF → HySEA `.grd`

No matter the input format, the following two-step pipeline **never fails**:

```
any_input.???  ──[Step 1: GMT or GDAL]──►  intermediate.tif
                                                │
                                                └──[Step 2: gmt grdconvert]=nf──►  hysea.grd
```

### Why this works

GMT's `grdconvert` with `=nf` output is the reference converter for HySEA.
GeoTIFF is the most universally supported raster format — virtually every GIS
tool (QGIS, ArcGIS, GDAL, GMT, Python/rasterio) can write it.
By passing through TIF you decouple the "read any format" problem from the
"write HySEA-compatible file" problem.

### Step 1 — Convert to GeoTIFF

**From Surfer Binary (v6 or v7) — using GDAL:**
```bash
gdal_translate your_grid.grd intermediate.tif
```

**From Surfer Binary — using GMT:**
```bash
gmt grdconvert your_grid.grd -Gintermediate.tif=tiff
```

**From any raster (QGIS):**
Raster → Save as → Format: GeoTIFF → CRS: EPSG:4326 (WGS84)

### Step 2 — TIF → HySEA `.grd`

```bash
gmt grdconvert intermediate.tif -Ghysea_ready.grd=nf
```

### One-liner for a batch of files

```bash
# Convert all Surfer .grd tiles to HySEA .grd via TIF
for i in $(seq 1 10); do
    gdal_translate dom${i}.grd dom${i}.tif
    gmt grdconvert dom${i}.tif -Gdom${i}_hysea.grd=nf
done
```

> **Tip:** If you are already in a GMT environment,
> `gmt grdconvert surfer.grd -Goutput.grd=nf` works directly
> (no TIF intermediate needed) — GMT auto-detects the Surfer format.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 12 — convert_via_tif(): the foolproof two-step pipeline
# Step 1: any format → GeoTIFF   (gdal_translate or gmt grdconvert)
# Step 2: GeoTIFF   → HySEA .grd (gmt grdconvert =nf)
# ─────────────────────────────────────────────────────────────────────────────

def _run(cmd):
    """Run a shell command; return (success, stderr)."""
    r = subprocess.run(cmd, capture_output=True, text=True)
    return r.returncode == 0, r.stderr.strip()


def convert_via_tif(input_file, output_file, verbose=True):
    """
    Convert any DTM format to HySEA .grd via an intermediate GeoTIFF.
    Requires GMT 6 and GDAL (gdal_translate) to be installed.

    This two-step route is format-agnostic:
      Step 1: input  → TIF   (gdal_translate; falls back to gmt grdconvert)
      Step 2: TIF    → .grd  (gmt grdconvert =nf)

    Parameters
    ----------
    input_file  : str — path to source DTM
    output_file : str — path for HySEA .grd output
    verbose     : bool
    """
    # Use a temporary TIF in the same folder as the output
    tif_tmp = output_file.replace('.grd', '_tmp.tif')

    try:
        # ── Step 1: → TIF ────────────────────────────────────────────────────
        ok, err = _run(['gdal_translate', input_file, tif_tmp])
        if not ok:
            if verbose:
                print(f'  gdal_translate failed: {err}')
                print('  Trying gmt grdconvert as fallback...')
            ok, err = _run(['gmt', 'grdconvert', input_file,
                            f'-G{tif_tmp}=tiff'])
            if not ok:
                raise RuntimeError(
                    f'Step 1 failed.\n'
                    f'  gdal_translate: not available or unsupported format\n'
                    f'  gmt grdconvert: {err}')

        if verbose:
            sz = os.path.getsize(tif_tmp) / 1024**2
            print(f'  Step 1 OK → {os.path.basename(tif_tmp)}  ({sz:.1f} MB)')

        # ── Step 2: TIF → HySEA .grd ─────────────────────────────────────────
        ok, err = _run(['gmt', 'grdconvert', tif_tmp,
                        f'-G{output_file}=nf'])
        if not ok:
            raise RuntimeError(f'Step 2 (gmt grdconvert =nf) failed: {err}')

        sz = os.path.getsize(output_file) / 1024**2
        if verbose:
            print(f'  Step 2 OK → {os.path.basename(output_file)}  ({sz:.1f} MB)')
            print(f'  ✅  Done')

    finally:
        # Always clean up the temporary TIF
        if os.path.exists(tif_tmp):
            os.remove(tif_tmp)


# ── Check tool availability ───────────────────────────────────────────────────
print('TOOL AVAILABILITY CHECK')
print('=' * 40)
for tool, args in [('gdal_translate', ['--version']),
                   ('gmt',            ['--version'])]:
    check = subprocess.run([tool] + args, capture_output=True, text=True)
    version = check.stdout.strip().split('\n')[0] if check.returncode == 0 else 'NOT FOUND'
    marker = '✅' if check.returncode == 0 else '❌'
    print(f'  {marker}  {tool:20s}  {version}')

print()
print('Usage example:')
print("  convert_via_tif('your_surfer6.grd', 'output_hysea.grd')")
print("  convert_via_tif('your_dem.tif',     'output_hysea.grd')")
print()
print('Batch example:')
print("  for i in range(1, 11):")
print("      convert_via_tif(f'dom{i}.grd', f'dom{i}_hysea.grd')")

---
## Summary

### Conversion functions

| Function | Input format | Key detail |
|----------|-------------|------------|
| `read_geotiff()` | GeoTIFF `.tif` | Uses TIFF tags 33550/33922; flips rows to ascending latitude |
| `read_surfer6_binary()` | Surfer v6 `DSBB` | Fixed header, float32 data, Surfer blank → NaN |
| `read_surfer7_binary()` | Surfer v7 `DSRB` | Tagged sections, float64 data, parses GRID + DATA |
| `convert_to_hysea_grd()` | Any of the above | Auto-detects format, optional sign flip |
| `gmt_convert()` | Any GMT-supported | Requires GMT 6 installed |
| `convert_via_tif()` | Any format | Two-step route via intermediate TIF; requires GMT 6 + GDAL |

### Checklist before using a converted grid in HySEA

```
□ verify_grd(output_file)        — check dimensions, extent, NaN count
□ plot_grd(output_file)          — visual sanity check
□ z < 0 in ocean areas           — confirm GMT sign convention
□ coordinates in decimal degrees — not UTM or projected metres
□ latitudes ascending (S → N)    — grdwrite always ensures this
□ no NaN values                  — NaN causes HySEA crash
```

### How to tell v6 from v7

```python
with open('file.grd', 'rb') as f:
    magic = f.read(4)
# b'DSBB' → Surfer 6     b'DSRB' → Surfer 7
```

Or from the shell:
```bash
xxd file.grd | head -1
file file.grd
```

### Next steps

- **[JN04](JN04_Grid_from_GEBCO.ipynb)** — Clip and resample a GEBCO GeoTIFF
  to a specific resolution and reference it in the HySEA parameter file.
- Forthcoming notebooks in this collection will cover **nested grid
  construction** (building finer/coarser grid hierarchies from converted data)
  and **parameter file setup** — check the `preprocessing/` folder of the
  HySEALab repository for updates.